# 11 金融计算模块 (core.financial) + 工具函数 (utils)

覆盖时间价值计算 / IRR / NPV / 数据集工具 / pandas扩展。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import hscredit as hsc
from hscredit import (
    germancredit, init_setting, seed_everything,
    fv, pv, pmt, nper, ipmt, ppmt, rate, npv, irr, mirr,
    load_pickle, save_pickle, feature_describe, groupby_feature_describe,
    round_float, init_setting,
)
init_setting()
print('模块加载成功')

## 1. 时间价值计算 (FV / PV / PMT / NPER / IPMT / PPMT / RATE)

In [ ]:
# 场景：年利率6%，月供，贷款10万，分3年还清
rate_m = 0.06 / 12  # 月利率
n_months = 36
loan = 100000

monthly_pmt = pmt(rate_m, n_months, -loan)
print(f'月供金额: {monthly_pmt:.2f} 元')

total_paid = monthly_pmt * n_months
print(f'总还款额: {total_paid:.2f} 元')
print(f'总利息:   {total_paid - loan:.2f} 元')

# 每月本金/利息拆分
print('\n前6期还款明细:')
for m in range(1, 7):
    interest = ipmt(rate_m, m, n_months, -loan)
    principal = ppmt(rate_m, m, n_months, -loan)
    print(f'  第{m:2d}期: 本金={principal:.2f}, 利息={interest:.2f}')

In [ ]:
# FV: 每月存2000，年利率4%，存5年到期多少
fv_val = fv(0.04/12, 60, -2000, 0)
print(f'FV (定投5年): {fv_val:.2f} 元')

# PV: 10年后需要100万，年利率5%，现在需要存多少
pv_val = pv(0.05/12, 120, 0, -1000000)
print(f'PV (目标100万): {pv_val:.2f} 元')

# NPER: 月供5000，年利率5%，还清20万需要多少期
nper_val = nper(0.05/12, 5000, -200000)
print(f'NPER (还清20万): {nper_val:.1f} 期 ({nper_val/12:.1f} 年)')

# RATE: 贷款50万，月供1万，36期，月利率
rate_val = rate(36, 10000, -500000)
print(f'月利率: {rate_val:.4%}, 年化: {rate_val*12:.4%}')

## 2. 投资决策计算 (NPV / IRR / MIRR)

In [ ]:
# 项目现金流：期初投入-100万，此后5年分别回收

In [ ]:
cashflows = [-1000000, 250000, 300000, 350000, 400000, 200000]
discount_rate = 0.10  # 折现率10%

npv_val = npv(discount_rate, cashflows)
print(f'NPV: {npv_val:,.2f} 元 ({'正值，项目可行' if npv_val > 0 else '负值，不建议投资'})')

irr_val = irr(cashflows)
print(f'IRR: {irr_val:.4%}')

mirr_val = mirr(cashflows, finance_rate=0.08, reinvest_rate=0.10)
print(f'MIRR: {mirr_val:.4%}')

## 3. seed_everything / round_float

In [ ]:
seed_everything(42)
print('随机种子已设置')

print(round_float(3.14159265, 3))
print(round_float(0.00001234, 6))

## 4. feature_describe / groupby_feature_describe

In [ ]:
df = germancredit()
feat_desc = feature_describe(df)
print(feat_desc.head(6))

In [ ]:
grp_desc = groupby_feature_describe(df, groupby='purpose', features=['credit.amount','duration.in.month'])
grp_desc.head(8)

## 5. save_pickle / load_pickle

In [ ]:
import tempfile, os
tmpdir = tempfile.mkdtemp()
obj = {'model': 'LogisticRegression', 'ks': 0.42, 'auc': 0.78}
pkl_path = os.path.join(tmpdir, 'model_meta.pkl')
save_pickle(obj, pkl_path)
loaded = load_pickle(pkl_path)
print('保存/加载成功:', loaded)

## 6. Pandas 扩展方法 (df.summary / df.save)

In [ ]:
import hscredit.utils.pandas_extensions  # 注册扩展方法
df_small = df.head(20)
# df.summary() — 快速数据概要
print(df_small.summary())

## 7. NumExprDerive — 表达式特征工程

In [ ]:
from hscredit import NumExprDerive
deriver = NumExprDerive(expressions={
    'loan_per_month': 'credit_amount / duration_in_month',
    'age_credit': 'age_in_years * credit_amount',
})
df2 = df.rename(columns={c: c.replace('.','_') for c in df.columns})
df_derived = deriver.fit_transform(df2)
print(df_derived[['loan_per_month','age_credit']].head(3))